# v2 — Reranqueamento com cross-encoder treinado por nós (mMARCO-pt)

Um único notebook, em 3 etapas, feito para o **Colab gratuito**:

| Etapa | O que faz | Checkpoint no Drive (`MyDrive/AssignmentPLN_v2/`) | Ao rodar de novo |
|---|---|---|---|
| **1. Dados** | 6.980 perguntas dev + top-100 do BM25 oficial; 300 mil triplas de treino; textos da coleção de 8,8 M | `data/` | **Pulada** se os dados já existem com a mesma configuração |
| **2. Treino** | Ajuste fino do nosso cross-encoder (laço PyTorch escrito à mão) | `checkpoints/<RUN_NAME>/`: ponto de retomada a cada 15 min, snapshots, `final/` | **Retoma** do último ponto; **pulada** se `final/` existe |
| **3. Avaliação** | BM25 vs CE pronto vs CE nosso; IC 95%, testes pareados, vitórias/derrotas, curva de aprendizado, latência | `results/` (pontuações em cache por modelo) | Modelos já pontuados **não** são recalculados |
| **4. Cascata + benchmark** | monoPTT5 reordena o top-N do CE; tabela passo a passo (qualidade, Δ, teto, latência) e cascata vs monoPTT5 direto | `results/t5_pair_scores.pkl` (cache por par) | Pares já pontuados **não** são recalculados |

**Como usar:** Ambiente de execução → Alterar o tipo de ambiente → **GPU T4** → **Executar tudo**.
Se a sessão cair: reconecte e **Executar tudo** de novo. Cada etapa continua de onde parou.

Dica para economizar a cota de GPU: a etapa 1 não usa GPU. Dá para rodá-la num ambiente **sem GPU**
(ela termina e o notebook para na etapa 2 com um aviso), depois trocar para T4 e Executar tudo.

Todas as configurações ficam na célula **Configuração** logo abaixo. Ver `phase2/README.md` e `phase2/WEAKNESSES.md`.

In [ ]:
!pip install -q ftfy==6.3.1

In [ ]:

# ==============================================================================
# Google Drive: tudo o que precisa sobreviver a uma desconexão fica aqui
# ==============================================================================
import os
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/AssignmentPLN_v2"
DATA = f"{BASE}/data"          # saída da etapa 1
CKPT = f"{BASE}/checkpoints"   # saída da etapa 2
RES  = f"{BASE}/results"       # saída das etapas 3 e 4
RAW  = "/content/mmarco"       # arquivos brutos do mMARCO (disco local da sessão, ~4,5 GB)
TMP  = "/content"
for d in (DATA, CKPT, RES, RAW):
    os.makedirs(d, exist_ok=True)

SEED = 42


In [ ]:
# ==============================================================================
# Configuração (única célula para editar)
# ==============================================================================
import json, math, time, random, shutil, platform, hashlib, gc
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import transformers
import ftfy
from torch.nn import BCEWithLogitsLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from huggingface_hub import hf_hub_download
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "nenhuma"
CC = torch.cuda.get_device_capability(0) if HAS_GPU else (0, 0)
USE_AMP = HAS_GPU            # T4: fp16 misto nos cross-encoders (Tensor Cores); monoPTT5 roda em fp32
print("GPU:", GPU_NAME, "| torch", torch.__version__, "| transformers", transformers.__version__)
if HAS_GPU and "T4" not in GPU_NAME:
    print("AVISO: este notebook foi dimensionado para a T4; tempos e latências não serão comparáveis.")

# ---------------- Etapa 1: dados ----------------
N_DEV_QUERIES   = None     # None = TODAS as perguntas do dev small (6.980), como nos artigos; um número = amostra aleatória
DEPTH           = 100      # candidatas do BM25 guardadas por pergunta (= maior K reranqueado)
N_TRAIN_TRIPLES = 300_000  # triplas de treino amostradas (a etapa 2 pode usar menos)
N_VAL_TRIPLES   = 2_000    # triplas separadas para validação do treino (NÃO são o conjunto de teste)
FIX_TEXT        = True     # corrige mojibake (ex.: "PlazaÃ®") com ftfy nas consultas e passagens

REPO = "unicamp-dl/mmarco"
FILES = {   # caminhos conferidos no repositório em 2026-11-24
    "collection":    "data/google/collections/portuguese_collection.tsv",   # 3,4 GB
    "train_queries": "data/google/queries/train/portuguese_queries.train.tsv",
    "dev_queries":   "data/google/queries/dev/portuguese_queries.dev.small.tsv",
    "dev_qrels":     "data/qrels.dev.small.tsv",
    "bm25_run":      "data/google/runs/run.bm25_portuguese-msmarco.txt",
    "triples":       "data/triples.train.ids.small.tsv",                    # só IDs, 0,9 GB
}

def baixar(key):
    t0 = time.time()
    p = hf_hub_download(REPO, FILES[key], repo_type="dataset", local_dir=RAW)
    print(f"  {key:14s} ok ({os.path.getsize(p)/1e6:,.0f} MB, {time.time()-t0:.0f}s)")
    return p

# ---------------- Etapa 2: treino ----------------
BASE_MODEL       = "nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large"
N_TRIPLES_USE    = 150_000   # triplas usadas (≤ N_TRAIN_TRIPLES). 150k = 300k pares
BATCH_TRIPLES    = 16        # 16 triplas = 32 pares por passo
LR               = 2e-5
WARMUP_FRAC      = 0.1
WEIGHT_DECAY     = 0.01
MAX_LEN_TRAIN    = 256       # passagens do MS MARCO são curtas; 256 tokens cobre quase todas
SNAPSHOT_FRACS   = [0.05, 0.1, 0.25, 0.5, 1.0]   # pontos da curva de aprendizado
RESUME_EVERY_MIN = 15
LOG_EVERY        = 100

RUN_NAME = f"mminilm_bce_{N_TRIPLES_USE//1000}k_lr{LR}_bs{2*BATCH_TRIPLES}"
RUN_DIR  = f"{CKPT}/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)

# ---------------- Etapa 3: avaliação ----------------
READY_MODEL        = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
K_VALUES           = [10, 25, 50, 100]
MAX_LEN_EVAL       = 512
FP16_INFER         = True    # aplicado igualmente a todos os modelos
EVAL_SNAPSHOTS     = True    # curva de aprendizado
N_SNAPSHOT_QUERIES = 1000    # snapshots avaliados numa subamostra fixa (a curva compara na MESMA subamostra)
N_BOOT             = 10_000
N_LAT_QUERIES      = 100

# ---------------- Etapa 4: cascata com monoPTT5 (3º estágio) ----------------
T5_MODEL      = "unicamp-dl/monoptt5-base"
T5_TEMPLATE   = "Pergunta: {query} Documento: {text} Relevante:"   # do model card
CASCADE_FROM  = ["CE nosso", "CE pronto"]   # quais CEs alimentam o monoPTT5 (tire um para economizar ~40% do tempo)
CASCADE_K     = 100                         # o CE reranqueia o top-100 do BM25...
CASCADE_NS    = [5, 10, 20]                 # ...e o monoPTT5 reranqueia só o top-N do CE (fatias do top-20)
T5_DIRECT     = True                        # comparação: monoPTT5 direto no top-100 do BM25, na subamostra SUB
T5_BATCH      = 32
T5_SAVE_EVERY_MIN = 10

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------------- Checkpoints: o que já está pronto no Drive ----------------
def dados_prontos():
    arquivos = ["dev_queries.parquet", "dev_passages.parquet", "train_triples.parquet", "meta.json"]
    if not all(os.path.exists(f"{DATA}/{a}") for a in arquivos):
        return False
    cfg = json.load(open(f"{DATA}/meta.json"))["config"]
    return cfg.get("N_DEV_QUERIES") == N_DEV_QUERIES and cfg.get("DEPTH") == DEPTH and cfg.get("FIX_TEXT") == FIX_TEXT

# As triplas de treino têm semente própria e não dependem do conjunto dev: se já existem, são mantidas.
PREP_TRAIN = not os.path.exists(f"{DATA}/train_triples.parquet")
DATA_DONE  = dados_prontos()
TRAIN_DONE = os.path.exists(f"{RUN_DIR}/final/config.json")
print(f"\nEtapa 1 (dados):  {'pronta, será pulada' if DATA_DONE else 'vai rodar' + ('' if PREP_TRAIN else ' (só a parte dev; triplas mantidas)')}")
print(f"Etapa 2 (treino): {'pronta, será pulada' if TRAIN_DONE else ('vai RETOMAR' if os.path.exists(f'{RUN_DIR}/resume.pt') else 'vai rodar do início')}  [{RUN_DIR}]")
print(f"Etapa 3 (avaliação): modelos já pontuados vêm do cache em {RES}")

---
# Etapa 1 — Dados (não precisa de GPU)

Corrige as duas fraquezas principais da v1 (`phase2/WEAKNESSES.md`):
**vazamento** (agora o conjunto oficial **dev small**, não perguntas de treino) e **corpus fácil** (agora as candidatas são o
**top-100 do BM25 oficial** sobre a coleção completa de 8,8 milhões, isto é, negativos difíceis por construção).
Também prepara as triplas de treino da etapa 2. Tempo estimado: 20–40 min (download de 3,4 GB).

In [ ]:
if not DATA_DONE:
    print("Baixando arquivos (disco local do Colab, não o Drive)...")
    paths = {k: baixar(k) for k in FILES if PREP_TRAIN or k not in ("triples", "train_queries")}
else:
    print("Etapa 1 já concluída (dados no Drive com a mesma configuração): pulada.")

## 1. Perguntas dev, rótulos (qrels) e candidatas do BM25 oficial

In [ ]:
if not DATA_DONE:
    def ler_tsv_dict(path):
        d = {}
        with open(path, encoding="utf-8") as f:
            for line in f:
                k, v = line.rstrip("\n").split("\t", 1)
                d[k] = v
        return d

    dev_queries = ler_tsv_dict(paths["dev_queries"])

    qrels = defaultdict(set)                       # formato TREC: qid 0 pid 1
    with open(paths["dev_qrels"]) as f:
        for line in f:
            qid, _, pid, rel = line.split()
            if int(rel) > 0:
                qrels[qid].add(pid)

    # 1ª passada no run: só descobrir quais qids existem (o arquivo tem ~7 milhões de linhas)
    run_qids = set()
    with open(paths["bm25_run"]) as f:
        for line in f:
            run_qids.add(line.split("\t", 1)[0])

    elegiveis = sorted(q for q in dev_queries if q in qrels and q in run_qids)
    print(f"dev small: {len(dev_queries)} perguntas | com qrels e no run: {len(elegiveis)}")

    if N_DEV_QUERIES is None or N_DEV_QUERIES >= len(elegiveis):
        amostra = set(elegiveis)                                               # dev small completo
    else:
        amostra = set(random.Random(SEED).sample(elegiveis, N_DEV_QUERIES))   # amostra aleatória
    print(f"perguntas avaliadas: {len(amostra)}")

    # 2ª passada: guardar o ranking completo (até 1000) só das perguntas amostradas
    ranking = defaultdict(list)
    with open(paths["bm25_run"]) as f:
        for line in f:
            qid, pid, rank = line.rstrip("\n").split("\t")
            if qid in amostra:
                ranking[qid].append((int(rank), pid))
    for qid in ranking:
        ranking[qid].sort()
    prof = [len(v) for v in ranking.values()]
    print(f"profundidade do run por pergunta: min={min(prof)} max={max(prof)}")

In [ ]:
if not DATA_DONE:
    dev_rows = []
    for qid in sorted(amostra):
        pids = [pid for _, pid in ranking[qid]]
        pos = {pid: r for r, pid in enumerate(pids, start=1)}
        rel_ranks = [pos.get(p, -1) for p in sorted(qrels[qid])]   # -1 = fora do top-1000 do BM25
        dev_rows.append({
            "qid": qid,
            "query": dev_queries[qid],
            "relevant_pids": sorted(qrels[qid]),
            "bm25_rel_ranks": rel_ranks,
            "candidates": pids[:DEPTH],
        })
    dev_df = pd.DataFrame(dev_rows)

    # Teto do reranqueamento = Recall@K do 1º estágio (fraqueza 7 da v1)
    def recall_at(ranks, k):
        return np.mean([sum(1 for r in rr if 0 < r <= k) / len(rr) for rr in ranks])
    def mrr10_bm25(ranks):
        return np.mean([max([1 / r for r in rr if 0 < r <= 10], default=0.0) for rr in ranks])

    print("BM25 oficial na amostra:")
    print(f"  MRR@10 = {mrr10_bm25(dev_df.bm25_rel_ranks):.4f}  (artigo mMARCO, dev completo, pt: 0.152)")
    for k in (10, 25, 50, 100, 1000):
        print(f"  Recall@{k:<4d} = {recall_at(dev_df.bm25_rel_ranks, k):.4f}   <- teto de um reranqueador com K={k}")
    print(f"perguntas com mais de 1 passagem relevante: {(dev_df.relevant_pids.str.len() > 1).sum()}")

## 2. Triplas de treino (amostra aleatória por *reservoir sampling*)

O arquivo tem dezenas de milhões de triplas `(qid, pid_positiva, pid_negativa)`. Lemos uma vez e mantemos uma
amostra aleatória uniforme de tamanho fixo, sem carregar tudo na memória. As negativas dessas triplas são
**negativos difíceis** (vieram do BM25), que é o que ensina um reranqueador (LEARN.md, Parte 5.5).

In [ ]:
if not DATA_DONE:
    if PREP_TRAIN:
        N_TOTAL = N_TRAIN_TRIPLES + N_VAL_TRIPLES
        rng = random.Random(SEED)
        reservoir, n_lidas = [], 0
        t0 = time.time()
        with open(paths["triples"]) as f:
            for line in f:
                n_lidas += 1
                if len(reservoir) < N_TOTAL:
                    reservoir.append(line)
                else:
                    j = rng.randrange(n_lidas)
                    if j < N_TOTAL:
                        reservoir[j] = line
        print(f"{n_lidas:,} triplas lidas em {time.time()-t0:.0f}s; amostradas {len(reservoir):,}")

        triples = [tuple(l.rstrip("\n").split("\t")) for l in reservoir]
        rng.shuffle(triples)
        val_triples, train_triples = triples[:N_VAL_TRIPLES], triples[N_VAL_TRIPLES:]
        del reservoir
    else:
        triples, train_triples, val_triples = [], [], []
        n_lidas = json.load(open(f"{DATA}/meta.json")).get("n_triples_in_file") if os.path.exists(f"{DATA}/meta.json") else None
        print("triplas de treino já preparadas: etapa pulada")

## 3. Textos: consultas de treino e passagens (uma passada pela coleção de 8,8 milhões)

In [ ]:
if not DATA_DONE:
    train_qids = {t[0] for t in triples}
    train_queries = {}
    if PREP_TRAIN:
        with open(paths["train_queries"], encoding="utf-8") as f:
            for line in f:
                qid, text = line.rstrip("\n").split("\t", 1)
                if qid in train_qids:
                    train_queries[qid] = text

    needed = set()
    for t in triples:
        needed.update(t[1:])
    for row in dev_rows:
        needed.update(row["candidates"]); needed.update(row["relevant_pids"])
    print(f"passagens necessárias: {len(needed):,}")

    passages, t0 = {}, time.time()
    with open(paths["collection"], encoding="utf-8", errors="replace") as f:   # 'replace' em vez de 'ignore': não some com bytes em silêncio
        for line in f:
            pid, text = line.rstrip("\n").split("\t", 1)
            if pid in needed:
                passages[pid] = text
    print(f"coleção lida em {time.time()-t0:.0f}s; encontradas {len(passages):,}")
    faltando = needed - passages.keys()
    assert not faltando, f"{len(faltando)} passagens não encontradas, ex.: {list(faltando)[:5]}"

In [ ]:
if not DATA_DONE:
    # Correção de codificação (fraqueza 12 da v1)
    n_fix = {"passagens": 0, "consultas": 0}
    if FIX_TEXT:
        for pid, txt in passages.items():
            novo = ftfy.fix_text(txt)
            if novo != txt:
                passages[pid] = novo; n_fix["passagens"] += 1
        for d in (train_queries,):
            for k, txt in d.items():
                novo = ftfy.fix_text(txt)
                if novo != txt:
                    d[k] = novo; n_fix["consultas"] += 1
        fixed_q = dev_df["query"].map(ftfy.fix_text)
        n_fix["consultas"] += int((fixed_q != dev_df["query"]).sum())
        dev_df["query"] = fixed_q
    print("textos alterados pelo ftfy:", n_fix)

In [ ]:
if not DATA_DONE:
    def para_df(trs, split):
        return pd.DataFrame({
            "qid": [t[0] for t in trs],
            "query": [train_queries[t[0]] for t in trs],
            "pos_text": [passages[t[1]] for t in trs],
            "neg_text": [passages[t[2]] for t in trs],
            "split": split,
        })
    if PREP_TRAIN:
        train_df = pd.concat([para_df(train_triples, "train"), para_df(val_triples, "val")], ignore_index=True)
        train_df.to_parquet(f"{DATA}/train_triples.parquet", index=False)
    else:
        print("train_triples.parquet já existe: mantido")

    dev_pids = set()
    for row in dev_rows:
        dev_pids.update(row["candidates"]); dev_pids.update(row["relevant_pids"])
    pass_df = pd.DataFrame({"pid": sorted(dev_pids)}).assign(text=lambda d: d.pid.map(passages))

    dev_df.to_parquet(f"{DATA}/dev_queries.parquet", index=False)
    pass_df.to_parquet(f"{DATA}/dev_passages.parquet", index=False)

    import torch, transformers
    meta = {
        "created": time.strftime("%Y-%m-%d %H:%M:%S"), "seed": SEED,
        "config": dict(N_DEV_QUERIES=N_DEV_QUERIES, DEPTH=DEPTH, N_TRAIN_TRIPLES=N_TRAIN_TRIPLES,
                       N_VAL_TRIPLES=N_VAL_TRIPLES, FIX_TEXT=FIX_TEXT),
        "files": FILES, "n_triples_in_file": n_lidas, "ftfy_changed": n_fix,
        "bm25_mrr10_sample": float(mrr10_bm25(dev_df.bm25_rel_ranks)),
        "bm25_recall": {k: float(recall_at(dev_df.bm25_rel_ranks, k)) for k in (10, 25, 50, 100, 1000)},
        "versions": {"python": platform.python_version(), "pandas": pd.__version__, "ftfy": ftfy.__version__,
                     "torch": torch.__version__, "transformers": transformers.__version__},
    }
    json.dump(meta, open(f"{DATA}/meta.json", "w"), indent=2, ensure_ascii=False)
    print(json.dumps(meta, indent=2, ensure_ascii=False))
    print("\nPronto. Arquivos em", DATA)

In [ ]:
# Libera a RAM da etapa 1 antes do treino
for _v in ["passages", "train_queries", "triples", "train_triples", "val_triples", "ranking", "train_df",
           "pass_df", "dev_rows", "dev_df", "run_qids", "needed"]:
    globals().pop(_v, None)
gc.collect()
DATA_DONE = dados_prontos()
print("Etapa 1 ok:", DATA_DONE)

---
# Etapa 2 — Treinar o nosso cross-encoder (GPU)

Ideia de **Nogueira & Cho (2019)**, **mMARCO** e **ptt5-v2**: ajustar o nosso próprio reranqueador nas triplas de treino do
mMARCO em português, partindo de `nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large`, o mesmo modelo base do
cross-encoder pronto. Mesma arquitetura, então a comparação isola **o efeito do treinamento**.
Cada tripla vira 2 pares rotulados, (consulta, positiva) → 1 e (consulta, negativa) → 0; perda = entropia cruzada binária.

Salva um ponto de retomada a cada `RESUME_EVERY_MIN` minutos e *snapshots* para a curva de aprendizado.
Tempo estimado (não medido): ~45 min para 150 mil triplas numa T4; a velocidade real é impressa a cada 100 passos.

In [ ]:
if not TRAIN_DONE:
    assert HAS_GPU, ("A etapa 2 precisa de GPU. Etapa 1 concluída e salva no Drive: troque para T4 "
                     "(Ambiente de execução -> Alterar o tipo de ambiente) e Executar tudo de novo.")

In [ ]:
if not TRAIN_DONE:
    df = pd.read_parquet(f"{DATA}/train_triples.parquet")
    tr = df[df.split == "train"].reset_index(drop=True)
    va = df[df.split == "val"].reset_index(drop=True)
    assert len(tr) >= N_TRIPLES_USE, f"só há {len(tr)} triplas preparadas"
    tr = tr.iloc[:N_TRIPLES_USE]   # a etapa 1 já embaralhou

    order = np.random.default_rng(SEED).permutation(len(tr))   # ordem fixa -> retomada determinística
    TOTAL_STEPS = len(tr) // BATCH_TRIPLES
    SNAPSHOT_STEPS = sorted({max(1, int(round(f * TOTAL_STEPS))) for f in SNAPSHOT_FRACS})
    print(f"treino: {len(tr):,} triplas = {2*len(tr):,} pares | {TOTAL_STEPS:,} passos | snapshots em {SNAPSHOT_STEPS}")
    print(f"validação: {len(va):,} triplas")

In [ ]:
if not TRAIN_DONE:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    # num_labels=1 -> cabeça nova, aleatória, que produz 1 logit (o aviso "newly initialized" é esperado)
    # .float(): o checkpoint base está salvo em fp16 (pytorch_model.bin = 236 MB ≈ 118M × 2 bytes) e versões novas
    # do transformers carregam no dtype salvo. Os pesos-mestre PRECISAM ser fp32: o autocast faz as contas em fp16
    # e o GradScaler só funciona com gradientes fp32 (senão: "Attempting to unscale FP16 gradients").
    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1).float().to(DEVICE)
    _dtypes = {p.dtype for p in model.parameters()}
    assert _dtypes == {torch.float32}, f"parâmetros não estão em fp32: {_dtypes}"
    print("parâmetros:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M", "| dtype:", _dtypes)

    no_decay = ("bias", "LayerNorm.weight")
    params = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(params, lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * TOTAL_STEPS), TOTAL_STEPS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)   # fp16 misto: ~2x mais rápido na T4
    loss_fn = BCEWithLogitsLoss()

    def encode(queries, passages):
        return tok(queries, passages, truncation=True, max_length=MAX_LEN_TRAIN, padding=True, return_tensors="pt").to(DEVICE)

    def make_batch(rows):
        n = len(rows)
        q = list(rows["query"]) * 2
        p = list(rows["pos_text"]) + list(rows["neg_text"])
        y = torch.tensor([1.0] * n + [0.0] * n, device=DEVICE)
        return encode(q, p), y

In [ ]:
if not TRAIN_DONE:
    @torch.no_grad()
    def validar(bs=64):
        """Acurácia por par (score(pos) > score(neg)) e perda média na validação."""
        model.eval()
        acertos, perdas = 0, []
        for i in range(0, len(va), bs):
            rows = va.iloc[i:i+bs]
            enc, y = make_batch(rows)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(**enc).logits.squeeze(-1).float()
            perdas.append(loss_fn(logits, y).item())
            n = len(rows)
            acertos += (logits[:n] > logits[n:]).sum().item()
        model.train()
        return acertos / len(va), float(np.mean(perdas))

    RESUME = f"{RUN_DIR}/resume.pt"
    CURVE  = f"{RUN_DIR}/curve.json"

    def salvar_retomada(step):
        tmp = os.path.join(TMP, "resume_tmp.pt")
        torch.save({"step": step, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state()}, tmp)
        shutil.copy(tmp, RESUME + ".part")      # grava no Drive sob outro nome e só então substitui:
        os.replace(RESUME + ".part", RESUME)    # uma queda no meio da cópia não corrompe a retomada anterior
        print(f"  [retomada salva no passo {step}]")

    def salvar_snapshot(step):
        """Snapshot leve (pesos em fp16, ~240 MB) para a curva de aprendizado."""
        acc, vloss = validar()
        sd = {k: v.detach().half().cpu() for k, v in model.state_dict().items()}
        torch.save(sd, f"{RUN_DIR}/snap_step{step}.pt")
        curve = json.load(open(CURVE)) if os.path.exists(CURVE) else []
        curve = [c for c in curve if c["step"] != step] + [{
            "step": step, "pairs_seen": step * 2 * BATCH_TRIPLES, "val_pair_acc": acc, "val_loss": vloss,
            "file": f"snap_step{step}.pt"}]
        json.dump(sorted(curve, key=lambda c: c["step"]), open(CURVE, "w"), indent=2)
        print(f"  [snapshot passo {step}: acurácia de par na validação = {acc:.4f}, perda = {vloss:.4f}]")

    start_step = 0
    if os.path.exists(RESUME):
        st = torch.load(RESUME, map_location=DEVICE, weights_only=False)
        model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
        scheduler.load_state_dict(st["scheduler"]); scaler.load_state_dict(st["scaler"])
        # map_location=DEVICE também levou os estados de RNG para a GPU; set_rng_state exige ByteTensor na CPU
        torch.set_rng_state(st["torch_rng"].cpu()); torch.cuda.set_rng_state(st["cuda_rng"].cpu())
        start_step = st["step"]
        print(f"Retomando do passo {start_step}/{TOTAL_STEPS}")
    else:
        acc0, l0 = validar()
        print(f"Antes do treino: acurácia de par = {acc0:.4f} (≈0,5 = acaso), perda = {l0:.4f}")

In [ ]:
if not TRAIN_DONE:
    model.train()
    t_start, t_last_resume, run_loss = time.time(), time.time(), []

    for step in range(start_step + 1, TOTAL_STEPS + 1):
        idx = order[(step - 1) * BATCH_TRIPLES: step * BATCH_TRIPLES]
        enc, y = make_batch(tr.iloc[idx])

        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(**enc).logits.squeeze(-1)
        loss = loss_fn(logits.float(), y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        run_loss.append(loss.item())

        if step % LOG_EVERY == 0:
            el = time.time() - t_start
            vel = (step - start_step) / el
            eta = (TOTAL_STEPS - step) / vel / 60
            print(f"passo {step:>6}/{TOTAL_STEPS} | perda {np.mean(run_loss):.4f} | lr {scheduler.get_last_lr()[0]:.2e} "
                  f"| {vel*2*BATCH_TRIPLES:,.0f} pares/s | faltam ~{eta:.0f} min")
            run_loss = []

        if step in SNAPSHOT_STEPS:
            salvar_snapshot(step)
        if time.time() - t_last_resume > RESUME_EVERY_MIN * 60 and step < TOTAL_STEPS:
            salvar_retomada(step); t_last_resume = time.time()

    print(f"Treino concluído em {(time.time()-t_start)/60:.1f} min (nesta sessão).")

In [ ]:
if not TRAIN_DONE:
    # Modelo final completo (fp32), carregável com from_pretrained na etapa 3
    FINAL = f"{RUN_DIR}/final"
    model.save_pretrained(FINAL); tok.save_pretrained(FINAL)
    json.dump({"base_model": BASE_MODEL, "N_TRIPLES_USE": N_TRIPLES_USE, "BATCH_TRIPLES": BATCH_TRIPLES, "LR": LR,
               "WARMUP_FRAC": WARMUP_FRAC, "WEIGHT_DECAY": WEIGHT_DECAY, "MAX_LEN_TRAIN": MAX_LEN_TRAIN, "TOTAL_STEPS": TOTAL_STEPS,
               "seed": SEED, "torch": torch.__version__, "transformers": transformers.__version__,
               "gpu": torch.cuda.get_device_name(0)},
              open(f"{RUN_DIR}/train_config.json", "w"), indent=2)
    if os.path.exists(RESUME):
        os.remove(RESUME)   # ~1,4 GB; não é mais necessário
    print("Salvo em", FINAL)
    print(json.dumps(json.load(open(CURVE)), indent=2))
else:
    print("Etapa 2 já concluída (modelo final no Drive): pulada.")

In [ ]:
# Libera a GPU antes da avaliação
for _v in ["model", "optimizer", "scheduler", "scaler", "tr", "va", "df", "st"]:
    globals().pop(_v, None)
gc.collect(); torch.cuda.empty_cache()
TRAIN_DONE = os.path.exists(f"{RUN_DIR}/final/config.json")
print("Etapa 2 ok:", TRAIN_DONE)

---
# Etapa 3 — Avaliação (GPU)

Compara, em **todas as perguntas do dev small** e nas **mesmas candidatas do BM25 oficial**:
**BM25** (ordem do run oficial), **CE pronto** (`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, o da v1) e **CE nosso** (etapa 2).
Cada par é pontuado **uma vez** (top-100) e os K menores são fatias disso, já que a pontuação é *pointwise*.
Inclui teto (Recall@K do BM25), IC 95% por bootstrap, Wilcoxon pareado com Holm, vitórias/derrotas, curva de aprendizado,
latência com aquecimento (p50/p95) e exemplos de vitórias **e** derrotas.

In [ ]:
assert HAS_GPU, "A etapa 3 precisa de GPU (T4)."
assert TRAIN_DONE, "Modelo final não encontrado: a etapa 2 não terminou."
dev  = pd.read_parquet(f"{DATA}/dev_queries.parquet")
P    = dict(pd.read_parquet(f"{DATA}/dev_passages.parquet").itertuples(index=False, name=None))
meta = json.load(open(f"{DATA}/meta.json"))
print(f"{len(dev)} perguntas dev | {len(P):,} passagens | profundidade {len(dev.candidates[0])}")
print("BM25 Recall@K (teto):", meta["bm25_recall"])

## 1. Pontuar as candidatas (uma vez por modelo, com cache no Drive)

In [ ]:
def carregar(nome):
    if nome == "CE pronto":
        tok = AutoTokenizer.from_pretrained(READY_MODEL)
        m = AutoModelForSequenceClassification.from_pretrained(READY_MODEL)
    elif nome == "CE nosso":
        tok = AutoTokenizer.from_pretrained(f"{RUN_DIR}/final")
        m = AutoModelForSequenceClassification.from_pretrained(f"{RUN_DIR}/final")
    else:   # snapshot: "snap_step{N}"
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)
        m = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)
        sd = torch.load(f"{RUN_DIR}/{nome}.pt", map_location="cpu")
        m.load_state_dict({k: v.float() for k, v in sd.items()})
    return tok, m.float().to(DEVICE).eval()   # fp32 para todos (o checkpoint base vem em fp16); o autocast decide a precisão

@torch.no_grad()
def pontuar_query(tok, m, query, pids):
    enc = tok([query] * len(pids), [P[p] for p in pids], truncation=True, max_length=MAX_LEN_EVAL,
              padding=True, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.float16, enabled=FP16_INFER and USE_AMP):
        return m(**enc).logits.squeeze(-1).float().cpu().numpy()

def pontuar_todos(nome, idx):
    """Pontua as candidatas das perguntas dev.iloc[idx]. Cache no Drive por modelo + conjunto exato de perguntas,
    então uma queda de sessão só perde o modelo em andamento."""
    tag = "CE_pronto" if nome == "CE pronto" else f"{RUN_NAME}_{nome.replace(' ', '_')}"
    chave = hashlib.md5(",".join(dev.qid.iloc[idx]).encode()).hexdigest()[:8]
    cache = f"{RES}/scores_{tag}_{len(idx)}q_{chave}.npy"
    if os.path.exists(cache):
        out = list(np.load(cache, allow_pickle=True)); assert len(out) == len(idx); return out
    tok, m = carregar(nome)
    t0 = time.time()
    out = [pontuar_query(tok, m, r.query, list(r.candidates)) for r in dev.iloc[idx].itertuples()]
    print(f"  {nome}: {time.time()-t0:.0f}s")
    np.save(cache, np.array(out, dtype=object), allow_pickle=True)
    del m; torch.cuda.empty_cache()
    return out

ALL = np.arange(len(dev))
SUB = np.sort(np.random.default_rng(SEED).choice(len(dev), size=min(N_SNAPSHOT_QUERIES, len(dev)), replace=False))
qidx = {"CE pronto": ALL, "CE nosso": ALL}                 # quais perguntas cada modelo pontuou
scores = {n: pontuar_todos(n, ALL) for n in qidx}
curve = json.load(open(f"{RUN_DIR}/curve.json"))
if EVAL_SNAPSHOTS:
    for c in curve:
        nome = c["file"][:-3]
        qidx[nome] = SUB
        scores[nome] = pontuar_todos(nome, SUB)

## 2. Métricas por pergunta, médias com IC 95% e teto

In [ ]:
def metricas(ranked, rel, k=10):
    top = ranked[:k]
    mrr = next((1 / i for i, p in enumerate(top, 1) if p in rel), 0.0)
    dcg = sum(1 / np.log2(i + 1) for i, p in enumerate(top, 1) if p in rel)
    idcg = sum(1 / np.log2(i + 1) for i in range(1, min(len(rel), k) + 1))
    rec = sum(1 for p in top if p in rel) / len(rel)
    return mrr, dcg / idcg, rec

def primeiro_rel(ranked, rel):
    return next((i for i, p in enumerate(ranked, 1) if p in rel), np.inf)

def reranquear(cands, s, K):
    o = np.argsort(-s[:K], kind="stable")     # empates mantêm a ordem do BM25
    return [cands[i] for i in o]

rels  = [set(r) for r in dev.relevant_pids]
cands = [list(c) for c in dev.candidates]
runs, ridx = {"BM25": cands}, {"BM25": ALL}   # nome -> rankings, e a quais perguntas eles se referem
for nome, sc in scores.items():
    for K in K_VALUES:
        runs[f"{nome} (K={K})"] = [reranquear(cands[i], s, K) for i, s in zip(qidx[nome], sc)]
        ridx[f"{nome} (K={K})"] = qidx[nome]
per_q = {n: np.array([metricas(rk, rels[i]) for rk, i in zip(lst, ridx[n])]) for n, lst in runs.items()}
first = {n: np.array([primeiro_rel(rk, rels[i]) for rk, i in zip(lst, ridx[n])]) for n, lst in runs.items()}

def ic95(x, chunk=500):
    """IC 95% por bootstrap, em blocos (10k x 6.980 índices de uma vez pesam ~0,5 GB na RAM do Colab)."""
    rng = np.random.default_rng(SEED)
    b = np.concatenate([x[rng.integers(0, len(x), size=(min(chunk, N_BOOT - s), len(x)))].mean(axis=1)
                        for s in range(0, N_BOOT, chunk)])
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

linhas = []
for nome, arr in per_q.items():
    lo, hi = ic95(arr[:, 0])
    linhas.append({"Sistema": nome, "perguntas": len(arr), "MRR@10": arr[:, 0].mean(), "IC95 MRR@10": f"[{lo:.4f}, {hi:.4f}]",
                   "nDCG@10": arr[:, 1].mean(), "Recall@10": arr[:, 2].mean()})
tab = pd.DataFrame(linhas)
principais = tab[~tab.Sistema.str.startswith("snap_")]
principais.to_csv(f"{RES}/tabela_resultados.csv", index=False)
display(principais.round(4))

teto = pd.DataFrame({"K": list(meta["bm25_recall"].keys()), "BM25 Recall@K (teto)": list(meta["bm25_recall"].values())})
teto.to_csv(f"{RES}/teto_bm25.csv", index=False); display(teto.round(4))

## 3. Significância: testes pareados por pergunta (K = 100)

Wilcoxon pareado sobre o MRR@10 de cada pergunta, mais IC 95% da **diferença** por bootstrap pareado.
Correção de Holm para as 3 comparações.

In [ ]:
K = 100
comparacoes = [("CE nosso", "BM25"), ("CE pronto", "BM25"), ("CE nosso", "CE pronto")]
def nm(s): return s if s == "BM25" else f"{s} (K={K})"

res = []
for a, b in comparacoes:
    xa, xb = per_q[nm(a)][:, 0], per_q[nm(b)][:, 0]
    d = xa - xb
    p = wilcoxon(xa, xb).pvalue if np.any(d != 0) else 1.0
    lo, hi = ic95(d)
    res.append({"A": nm(a), "B": nm(b), "ΔMRR@10 (A−B)": d.mean(), "IC95 Δ": f"[{lo:.4f}, {hi:.4f}]", "p (Wilcoxon)": p})
sig = pd.DataFrame(res)
# Holm
ordem = np.argsort(sig["p (Wilcoxon)"].values)
m = len(sig); p_holm = np.empty(m); acc = 0
for r, i in enumerate(ordem):
    acc = max(acc, min(1.0, (m - r) * sig["p (Wilcoxon)"].values[i])); p_holm[i] = acc
sig["p (Holm)"] = p_holm
sig["significativo (α=0,05)"] = sig["p (Holm)"] < 0.05
sig.to_csv(f"{RES}/significancia.csv", index=False)
display(sig)

## 4. Vitórias, empates e derrotas por pergunta (K = 100)

Compara a posição da **primeira passagem relevante**. *Resgate* = estava fora do top 10 e entrou;
*perda* = estava no top 10 e saiu. (Algumas "derrotas" podem ser falsos negativos: passagens relevantes não rotuladas.)

In [ ]:
wtl = []
for a, b in comparacoes:
    ra, rb = first[nm(a)], first[nm(b)]
    wtl.append({"A": nm(a), "B": nm(b),
                "vitórias A": int((ra < rb).sum()), "empates": int((ra == rb).sum()), "derrotas A": int((ra > rb).sum()),
                "resgates (B>10 → A≤10)": int(((rb > 10) & (ra <= 10)).sum()),
                "perdas (B≤10 → A>10)": int(((rb <= 10) & (ra > 10)).sum())})
wtl = pd.DataFrame(wtl); wtl.to_csv(f"{RES}/vitorias_derrotas.csv", index=False); display(wtl)

def r2(x): return -1 if np.isinf(x) else int(x)
ranks_df = pd.DataFrame({"qid": dev.qid, "query": dev["query"],
                         **{f"rank_{n}": [r2(x) for x in first[n]] for n in ["BM25", nm("CE pronto"), nm("CE nosso")]}})
ranks_df.to_csv(f"{RES}/ranks_por_pergunta.csv", index=False)

## 5. Curva de aprendizado do nosso modelo

In [ ]:
if EVAL_SNAPSHOTS:
    cv = []
    for c in curve:
        arr = per_q[f"{c['file'][:-3]} (K=100)"][:, 0]
        lo, hi = ic95(arr)
        cv.append({**c, "MRR@10 (K=100)": arr.mean(), "ic_lo": lo, "ic_hi": hi})
    cv = pd.DataFrame(cv); cv.to_csv(f"{RES}/curva_aprendizado.csv", index=False)
    display(cv[["pairs_seen", "val_pair_acc", "MRR@10 (K=100)", "ic_lo", "ic_hi"]].round(4))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(cv.pairs_seen, cv["MRR@10 (K=100)"], yerr=[cv["MRR@10 (K=100)"] - cv.ic_lo, cv.ic_hi - cv["MRR@10 (K=100)"]],
                marker="o", capsize=3, label="CE nosso (snapshots)")
    ax.axhline(per_q[nm("CE pronto")][SUB, 0].mean(), ls="--", c="gray", label="CE pronto (mesma subamostra)")
    ax.axhline(per_q["BM25"][SUB, 0].mean(), ls=":", c="black", label="BM25 (mesma subamostra)")
    ax.set_title(f"Curva de aprendizado ({len(SUB)} perguntas dev)")
    ax.set_xscale("log"); ax.set_xlabel("pares de treino vistos"); ax.set_ylabel("MRR@10 (dev, K=100)")
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(f"{RES}/curva_aprendizado.png", dpi=200); plt.show()

## 6. Latência (com aquecimento, mesma GPU, p50/p95)

Mede só o reranqueador (tokenização + modelo), por pergunta. O 1º estágio vem de um run pré-calculado,
então a latência do BM25 **não** é medida aqui (num sistema real, Pyserini/Lucene responde em milissegundos).

In [ ]:
lat = []
sub = dev.sample(N_LAT_QUERIES, random_state=SEED)
for nome in ["CE pronto", "CE nosso"]:
    tok, m = carregar(nome)
    for r in sub.head(10).itertuples():              # aquecimento
        pontuar_query(tok, m, r.query, list(r.candidates))
    for K in K_VALUES:
        ts = []
        for r in sub.itertuples():
            torch.cuda.synchronize(); t0 = time.perf_counter()
            pontuar_query(tok, m, r.query, list(r.candidates)[:K])
            torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
        lat.append({"Modelo": nome, "K": K, "média ms": np.mean(ts), "p50 ms": np.percentile(ts, 50),
                    "p95 ms": np.percentile(ts, 95), "ms por par": np.mean(ts) / K})
    del m; torch.cuda.empty_cache()
lat = pd.DataFrame(lat); lat.to_csv(f"{RES}/latencia.csv", index=False); display(lat.round(2))

## 7. Exemplos qualitativos: vitórias **e** derrotas do nosso modelo sobre o BM25 (K = 100)

In [ ]:
rb, ro = first["BM25"], first[nm("CE nosso")]
vit = np.where((rb > 10) & (ro == 1))[0]
der = np.where((rb <= 3) & (ro > rb))[0]
rnd = random.Random(SEED)
def mostrar(i, titulo):
    row = dev.iloc[i]
    top1 = runs[nm("CE nosso")][i][0]
    print("=" * 100); print(f"{titulo} | qid {row.qid} | BM25: {r2(rb[i])} -> CE nosso: {r2(ro[i])} | CE pronto: {r2(first[nm('CE pronto')][i])}")
    print("PERGUNTA:", row.query)
    print("\nRELEVANTE (rótulo):", P[row.relevant_pids[0]][:600])
    if top1 not in set(row.relevant_pids):
        print("\nTOP-1 DO CE NOSSO (não rotulada como relevante):", P[top1][:600])
print(f"vitórias fortes (BM25>10 -> 1º): {len(vit)} | derrotas (BM25≤3 e piorou): {len(der)}\n")
for i in rnd.sample(list(vit), min(3, len(vit))): mostrar(i, "VITÓRIA")
for i in rnd.sample(list(der), min(3, len(der))): mostrar(i, "DERROTA")

In [ ]:
env = {"torch": torch.__version__, "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0),
       "FP16_INFER": FP16_INFER, "USE_AMP": USE_AMP, "compute_capability": list(CC), "MAX_LEN_EVAL": MAX_LEN_EVAL, "RUN_NAME": RUN_NAME, "N_BOOT": N_BOOT, "seed": SEED,
       "data_meta": meta}
json.dump(env, open(f"{RES}/env.json", "w"), indent=2, ensure_ascii=False)
print("Resultados em", RES)

---
# Etapa 4 — Terceiro estágio (cascata) e benchmark passo a passo

Ideia de **Multi-Stage Document Ranking** (BM25 → monoBERT → duoBERT) e **ptt5-v2**: um funil em que cada estágio é mais
caro e vê menos candidatas.

```
BM25 (top-100)  →  cross-encoder (reordena 100)  →  monoPTT5 (reordena só o top-N do CE)  →  top-10
```

**monoPTT5** (`unicamp-dl/monoptt5-base`) é um T5 (encoder + decoder) ajustado como reranqueador em português e inglês
(mMARCO). Ele não tem cabeça de pontuação: recebe `Pergunta: … Documento: … Relevante:` e a pontuação é
log P("Sim") contra P("Não") no **primeiro token gerado**. Roda em **fp32** (T5 em fp16 costuma estourar).

O benchmark mede, **para cada passo do pipeline**: qualidade (MRR@10 com IC 95%), ganho sobre o passo anterior com teste
pareado, o **teto** que o passo seguinte herda (Recall@N na passagem de bastão) e a latência acumulada.

Pontuações do monoPTT5 são guardadas **por par (pergunta, passagem)** no Drive: o mesmo par vindo de CEs diferentes é
pontuado uma vez, e uma queda de sessão perde no máximo `T5_SAVE_EVERY_MIN` minutos.
Custo estimado (não medido): ~150–270 mil pares, 1–2 h numa T4; a célula imprime a velocidade real.

In [ ]:
import pickle
from transformers import T5ForConditionalGeneration
assert HAS_GPU, "A etapa 4 precisa de GPU."

t5_tok = AutoTokenizer.from_pretrained(T5_MODEL)
t5 = T5ForConditionalGeneration.from_pretrained(T5_MODEL).float().to(DEVICE).eval()
ID_SIM, ID_NAO = t5_tok.convert_tokens_to_ids("▁Sim"), t5_tok.convert_tokens_to_ids("▁Não")
assert (ID_SIM, ID_NAO) == (4295, 852), (ID_SIM, ID_NAO)   # ids conferidos no spiece.vocab do modelo (2026-11-24)

@torch.no_grad()
def t5_pontuar_textos(query, textos, bs=T5_BATCH):
    out = []
    for i in range(0, len(textos), bs):
        inp = [T5_TEMPLATE.format(query=query, text=t) for t in textos[i:i+bs]]
        enc = t5_tok(inp, truncation=True, max_length=MAX_LEN_EVAL, padding=True, return_tensors="pt").to(DEVICE)
        dec = torch.full((enc.input_ids.shape[0], 1), t5.config.decoder_start_token_id, device=DEVICE)
        logits = t5(**enc, decoder_input_ids=dec).logits[:, 0, [ID_SIM, ID_NAO]].float()
        out.append(torch.log_softmax(logits, dim=-1)[:, 0].cpu().numpy())   # log P(Sim | {Sim, Não})
    return np.concatenate(out)

# Autoteste com o exemplo do model card: o card mostra 0.8187 (relevante) e 0.0080 (não relevante)
_s = np.exp(t5_pontuar_textos("O futebol é uma paixão nacional",
        ["O futebol é superestimado e não deveria receber tanta atenção.",
         "O futebol é uma parte essencial da cultura brasileira e une as pessoas."]))
print("autoteste monoPTT5:", np.round(_s, 4), "(model card: [0.0080, 0.8187])")
assert _s[1] > _s[0] and abs(_s[1] - 0.8187) < 0.05 and abs(_s[0] - 0.0080) < 0.05, "monoPTT5 não reproduz o model card"

In [ ]:
T5_CACHE = f"{RES}/t5_pair_scores.pkl"     # {(qid, pid): log P(Sim)}
t5_cache = pickle.load(open(T5_CACHE, "rb")) if os.path.exists(T5_CACHE) else {}
print(f"pares no cache do monoPTT5: {len(t5_cache):,}")

def t5_salvar():
    tmp = os.path.join(TMP, "t5_cache_tmp.pkl")
    pickle.dump(t5_cache, open(tmp, "wb"))
    shutil.copy(tmp, T5_CACHE + ".part"); os.replace(T5_CACHE + ".part", T5_CACHE)

def t5_garantir(pedidos, rotulo):
    """pedidos: lista de (índice da pergunta, [pids]). Pontua só os pares que ainda não estão no cache."""
    faltam = [(i, [p for p in ps if (dev.qid.iloc[i], p) not in t5_cache]) for i, ps in pedidos]
    faltam = [(i, ps) for i, ps in faltam if ps]
    total = sum(len(ps) for _, ps in faltam)
    print(f"{rotulo}: {total:,} pares novos para pontuar")
    t0 = t_save = time.time(); feitos = 0
    for j, (i, ps) in enumerate(faltam, 1):
        qid = dev.qid.iloc[i]
        for p, v in zip(ps, t5_pontuar_textos(dev["query"].iloc[i], [P[p] for p in ps])):
            t5_cache[(qid, p)] = float(v)
        feitos += len(ps)
        if j % 500 == 0:
            vel = feitos / (time.time() - t0)
            print(f"  {feitos:,}/{total:,} pares | {vel:.0f} pares/s | faltam ~{(total-feitos)/vel/60:.0f} min")
        if time.time() - t_save > T5_SAVE_EVERY_MIN * 60:
            t5_salvar(); t_save = time.time()
    t5_salvar()

NMAX = max(CASCADE_NS)
for ce in CASCADE_FROM:
    lst = runs[f"{ce} (K={CASCADE_K})"]
    t5_garantir([(i, lst[i][:NMAX]) for i in ALL], f"cascata a partir de {ce} (top-{NMAX})")
if T5_DIRECT:
    t5_garantir([(i, cands[i]) for i in SUB], f"monoPTT5 direto no top-100 do BM25 ({len(SUB)} perguntas)")

In [ ]:
def cascata(rank_ce, qid, N):
    """monoPTT5 reordena o top-N do CE; o resto da lista do CE continua atrás (importa quando N < 10)."""
    top = rank_ce[:N]
    s = np.array([t5_cache[(qid, p)] for p in top])
    return [top[k] for k in np.argsort(-s, kind="stable")] + rank_ce[N:]

def registrar(nome, rankings, idx):
    runs[nome], ridx[nome] = rankings, idx
    per_q[nome] = np.array([metricas(rk, rels[i]) for rk, i in zip(rankings, idx)])
    first[nome] = np.array([primeiro_rel(rk, rels[i]) for rk, i in zip(rankings, idx)])

for ce in CASCADE_FROM:
    base = runs[f"{ce} (K={CASCADE_K})"]
    for N in CASCADE_NS:
        registrar(f"{ce} (K={CASCADE_K}) → monoPTT5 (N={N})",
                  [cascata(base[i], dev.qid.iloc[i], N) for i in ALL], ALL)
if T5_DIRECT:
    registrar("monoPTT5 (K=100)",
              [cascata(cands[i], dev.qid.iloc[i], len(cands[i])) for i in SUB], SUB)

def recall_n(rankings, idx, n):
    """Fração das relevantes que está no top-n: o teto que o próximo estágio herda."""
    return float(np.mean([len(set(rk[:n]) & rels[i]) / len(rels[i]) for rk, i in zip(rankings, idx)]))

In [ ]:
# Latência do monoPTT5 (com aquecimento, mesma GPU), por N
lat_t5, sub_lat = [], dev.sample(N_LAT_QUERIES, random_state=SEED)
Ns_lat = sorted(set(CASCADE_NS + ([100] if T5_DIRECT else [])))
for r in sub_lat.head(5).itertuples():
    t5_pontuar_textos(r.query, [P[p] for p in list(r.candidates)[:10]])
for N in Ns_lat:
    ts = []
    for r in sub_lat.itertuples():
        textos = [P[p] for p in list(r.candidates)[:N]]
        torch.cuda.synchronize(); t0 = time.perf_counter()
        t5_pontuar_textos(r.query, textos)
        torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
    lat_t5.append({"Modelo": "monoPTT5", "N": N, "média ms": np.mean(ts), "p50 ms": np.percentile(ts, 50),
                   "p95 ms": np.percentile(ts, 95), "ms por par": np.mean(ts) / N})
lat_t5 = pd.DataFrame(lat_t5); lat_t5.to_csv(f"{RES}/latencia_monoptt5.csv", index=False); display(lat_t5.round(2))

def ms_ce(nome, K):  return float(lat.query("Modelo == @nome and K == @K")["média ms"].iloc[0])
def ms_t5(N):        return float(lat_t5.query("N == @N")["média ms"].iloc[0])

## Benchmark passo a passo (todas as perguntas dev)

Cada linha acrescenta **um** passo ao pipeline. `Δ` e `p` comparam com a linha **anterior do mesmo caminho**
(Wilcoxon pareado, Holm sobre todas as comparações da tabela). `Teto p/ próximo` = Recall@N da saída deste passo,
isto é, o máximo que o passo seguinte pode alcançar. A latência do BM25 não entra (run pré-calculado).

In [ ]:
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); acc = 0.0
    for r, i in enumerate(o):
        acc = max(acc, min(1.0, (len(ps) - r) * ps[i])); adj[i] = acc
    return adj

linhas = []
for ce in CASCADE_FROM:
    caminho = [("0. BM25 (top-100)", "BM25", None, 0.0),
               (f"1. + {ce} (K={CASCADE_K})", f"{ce} (K={CASCADE_K})", NMAX, ms_ce(ce, CASCADE_K))]
    caminho += [(f"2. + monoPTT5 (N={N})", f"{ce} (K={CASCADE_K}) → monoPTT5 (N={N})", None,
                 ms_ce(ce, CASCADE_K) + ms_t5(N)) for N in CASCADE_NS]
    anterior = None
    for passo, nome, n_teto, ms in caminho:
        x = per_q[nome][:, 0]; lo, hi = ic95(x)
        ref = "BM25" if passo.startswith("1.") else (f"{ce} (K={CASCADE_K})" if passo.startswith("2.") else None)
        lin = {"caminho": ce, "passo": passo, "MRR@10": x.mean(), "IC95": f"[{lo:.4f}, {hi:.4f}]",
               "nDCG@10": per_q[nome][:, 1].mean(), "Recall@10": per_q[nome][:, 2].mean(),
               "Teto p/ próximo": recall_n(runs[nome], ridx[nome], NMAX) if passo.startswith(("0.", "1.")) else np.nan,
               "latência rerank ms": ms, "Δ MRR@10": np.nan, "p": np.nan}
        if passo.startswith("0."):
            lin["Teto p/ próximo"] = meta["bm25_recall"][str(CASCADE_K)] if str(CASCADE_K) in meta["bm25_recall"] else recall_n(runs[nome], ridx[nome], CASCADE_K)
        if ref:
            d = x - per_q[ref][:, 0]
            lin["Δ MRR@10"] = d.mean()
            lin["p"] = wilcoxon(x, per_q[ref][:, 0]).pvalue if np.any(d != 0) else 1.0
        linhas.append(lin)
bench = pd.DataFrame(linhas)
m = bench.p.notna(); bench.loc[m, "p (Holm)"] = holm(bench.loc[m, "p"])
bench.to_csv(f"{RES}/benchmark_passo_a_passo.csv", index=False)
display(bench.round(4))
print("Teto: no passo 0 é Recall@100 do BM25 (o que o CE pode alcançar); no passo 1 é Recall@"
      f"{NMAX} do CE (o que o monoPTT5 com N={NMAX} pode alcançar).")

## Cascata vs usar o modelo grande direto (subamostra `SUB`)

A pergunta prática de um funil: **o CE antes do monoPTT5 economiza tempo sem perder qualidade?**
Compara, nas mesmas perguntas, o monoPTT5 reranqueando direto o top-100 do BM25 contra a cascata.

In [ ]:
if T5_DIRECT:
    pos = {i: k for k, i in enumerate(ALL)}          # SUB indexa direto os arrays de "todas as perguntas"
    def mrr_sub(nome):
        arr = per_q[nome][:, 0]
        return arr if len(arr) == len(SUB) else arr[SUB]
    linhas = [("BM25", "BM25", 0.0),
              (f"CE nosso (K={CASCADE_K})", f"CE nosso (K={CASCADE_K})", ms_ce("CE nosso", CASCADE_K))]
    linhas += [(f"CE nosso → monoPTT5 (N={N})", f"CE nosso (K={CASCADE_K}) → monoPTT5 (N={N})",
                ms_ce("CE nosso", CASCADE_K) + ms_t5(N)) for N in CASCADE_NS]
    linhas += [("monoPTT5 direto (K=100)", "monoPTT5 (K=100)", ms_t5(100))]
    ref = mrr_sub("monoPTT5 (K=100)")
    tab_sub = []
    for rot, nome, ms in linhas:
        x = mrr_sub(nome); lo, hi = ic95(x); d = x - ref
        tab_sub.append({"sistema": rot, "MRR@10": x.mean(), "IC95": f"[{lo:.4f}, {hi:.4f}]",
                        "Δ vs monoPTT5 direto": d.mean(),
                        "p vs direto": (wilcoxon(x, ref).pvalue if np.any(d != 0) else 1.0),
                        "latência rerank ms": ms})
    tab_sub = pd.DataFrame(tab_sub); tab_sub.to_csv(f"{RES}/cascata_vs_direto_subamostra.csv", index=False)
    print(f"{len(SUB)} perguntas (subamostra SUB)"); display(tab_sub.round(4))

In [ ]:
# Vitórias / derrotas do 3º estágio sobre o 2º, e exemplos
for ce in CASCADE_FROM:
    a, b = f"{ce} (K={CASCADE_K}) → monoPTT5 (N={NMAX})", f"{ce} (K={CASCADE_K})"
    ra, rb_ = first[a], first[b]
    print(f"{ce}: monoPTT5 (N={NMAX}) vs CE  ->  vitórias {int((ra < rb_).sum())} | empates {int((ra == rb_).sum())} "
          f"| derrotas {int((ra > rb_).sum())}")
pd.DataFrame({"qid": dev.qid, "query": dev["query"],
              **{f"rank_{n}": [r2(x) for x in first[n]] for n in first if len(first[n]) == len(dev)}}
             ).to_csv(f"{RES}/ranks_por_pergunta_com_cascata.csv", index=False)
json.dump({"T5_MODEL": T5_MODEL, "T5_TEMPLATE": T5_TEMPLATE, "CASCADE_K": CASCADE_K, "CASCADE_NS": CASCADE_NS,
           "CASCADE_FROM": CASCADE_FROM, "t5_dtype": "float32", "gpu": GPU_NAME, "USE_AMP_CE": USE_AMP},
          open(f"{RES}/env_etapa4.json", "w"), indent=2, ensure_ascii=False)
print("Etapa 4 salva em", RES)

---
# Etapa 5 — Fusão de pontuações (sem GPU)

Na etapa 4, o monoPTT5 por cima do CE pronto não mudou a média, mas mudou ~2 mil perguntas (1.083 vitórias × 1.082 derrotas):
os dois modelos têm força parecida e **erram em perguntas diferentes**. Em vez de um **substituir** a ordem do outro (cascata),
aqui os dois **votam juntos** (ideia do survey de Lin, Nogueira & Yates; RRF).

- **Interpolação:** `α·z(fonte 1) + (1−α)·z(fonte 2)`, com z-score por pergunta (as escalas não são comparáveis).
  α = 1 é a fonte 1 sozinha; α = 0 é a fonte 2 sozinha (para CE + monoPTT5, isso é a própria cascata).
  **α é escolhido por validação cruzada em 2 metades** das perguntas, então nenhuma pergunta é avaliada com um α escolhido nela.
  A linha `[otimista]` (melhor α no próprio teste) é só referência e não deve ir para o artigo como resultado.
- **RRF** (*Reciprocal Rank Fusion*, k = 60): soma `1/(k + posição)` de cada fonte. Sem parâmetro ajustado.
- **Pools:** com o monoPTT5, a fusão acontece no top-20 do CE (é onde há pontuação do T5); sem ele, no top-100 inteiro.
- **BM25:** o run oficial só traz posições, então o BM25 entra como `−log(1 + posição)`.

**Como rodar sozinha** (por exemplo, numa sessão nova sem GPU): execute as células **Drive** e **Configuração** e depois esta.
Ela lê só os caches das etapas 3 e 4 no Drive. Leva poucos minutos.

In [ ]:
# ==============================================================================
# Etapa 5 — Fusão de pontuações (não precisa de GPU: usa só os caches do Drive)
# Requer: células "Drive" e "Configuração" executadas; etapas 3 e 4 concluídas antes (pontuações no Drive).
# ==============================================================================
import pickle, hashlib

FUSION_N = 20                               # tamanho do pool quando o monoPTT5 participa (ele só pontuou o top-20 de cada CE)
ALPHAS   = np.round(np.linspace(0, 1, 11), 2)
RRF_K    = 60

dev_f   = pd.read_parquet(f"{DATA}/dev_queries.parquet")
cands_f = [list(c) for c in dev_f.candidates]
rels_f  = [set(r) for r in dev_f.relevant_pids]
qids_f  = list(dev_f.qid)
nq      = len(dev_f)
_chave  = hashlib.md5(",".join(dev_f.qid).encode()).hexdigest()[:8]   # mesma chave de cache da etapa 3

def _scores(tag):
    f = f"{RES}/scores_{tag}_{nq}q_{_chave}.npy"
    assert os.path.exists(f), f"cache não encontrado (rode a etapa 3): {f}"
    return list(np.load(f, allow_pickle=True))
S  = {"CE pronto": _scores("CE_pronto"), "CE nosso": _scores(f"{RUN_NAME}_CE_nosso")}
T5 = pickle.load(open(f"{RES}/t5_pair_scores.pkl", "rb"))
print(f"{nq} perguntas | pontuações: {list(S)} + monoPTT5 ({len(T5):,} pares)")

def _metricas(rk, rel, k=10):
    top = rk[:k]
    mrr = next((1 / i for i, p in enumerate(top, 1) if p in rel), 0.0)
    dcg = sum(1 / np.log2(i + 1) for i, p in enumerate(top, 1) if p in rel)
    idcg = sum(1 / np.log2(i + 1) for i in range(1, min(len(rel), k) + 1))
    return mrr, dcg / idcg, sum(1 for p in top if p in rel) / len(rel)

def _z(x):
    """Normaliza por pergunta: as escalas (logit do CE, log P(Sim) do T5, posição do BM25) não são comparáveis."""
    x = np.asarray(x, float); s = x.std()
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)

def _rrf(*fontes):
    """Reciprocal Rank Fusion: só usa as posições, então dispensa normalização."""
    tot = 0.0
    for x in fontes:
        r = np.empty(len(x)); r[np.argsort(-np.asarray(x), kind="stable")] = np.arange(1, len(x) + 1)
        tot = tot + 1.0 / (RRF_K + r)
    return tot

def _fonte(nome, i, pool, pos):
    if nome in S:      return np.asarray(S[nome][i])[[pos[p] for p in pool]]
    if nome == "T5":   return np.array([T5[(qids_f[i], p)] for p in pool])
    if nome == "BM25": return -np.log1p([pos[p] for p in pool])   # o run oficial só traz posições, não pontuações
    raise ValueError(nome)

# fontes[0] é a "base": define o pool (top-N dela) e a ordem da cauda fora do pool
CONFIGS = {
    "CE pronto + monoPTT5":            dict(base="CE pronto", N=FUSION_N, fontes=["CE pronto", "T5"]),
    "CE nosso + monoPTT5":             dict(base="CE nosso",  N=FUSION_N, fontes=["CE nosso", "T5"]),
    "CE pronto + CE nosso":            dict(base="CE pronto", N=100,      fontes=["CE pronto", "CE nosso"]),
    "CE pronto + BM25":                dict(base="CE pronto", N=100,      fontes=["CE pronto", "BM25"]),
    "CE nosso + BM25":                 dict(base="CE nosso",  N=100,      fontes=["CE nosso", "BM25"]),
    "CE pronto + CE nosso + monoPTT5": dict(base="CE pronto", N=FUSION_N, fontes=["CE pronto", "CE nosso", "T5"]),
}

def _preparar(cfg):
    out = []
    for i in range(nq):
        c = cands_f[i]; pos = {p: k for k, p in enumerate(c)}
        base = [c[k] for k in np.argsort(-np.asarray(S[cfg["base"]][i]), kind="stable")]
        pool, cauda = base[:cfg["N"]], base[cfg["N"]:]
        out.append((pool, cauda, [_fonte(f, i, pool, pos) for f in cfg["fontes"]]))
    return out

def _avaliar(prep, combinar):
    res = np.empty((nq, 3))
    for i, (pool, cauda, fs) in enumerate(prep):
        rk = [pool[k] for k in np.argsort(-combinar(fs), kind="stable")] + cauda
        res[i] = _metricas(rk, rels_f[i])
    return res

def _ic(x):
    rng = np.random.default_rng(SEED)
    b = np.concatenate([x[rng.integers(0, len(x), size=(min(500, N_BOOT - s), len(x)))].mean(axis=1)
                        for s in range(0, N_BOOT, 500)])
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

def _holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); acc = 0.0
    for r, i in enumerate(o):
        acc = max(acc, min(1.0, (len(ps) - r) * ps[i])); adj[i] = acc
    return adj

# α é escolhido por validação cruzada em 2 metades: nenhuma pergunta é avaliada com um α escolhido nela mesma
_perm = np.random.default_rng(SEED).permutation(nq)
FOLDS = [np.sort(_perm[: nq // 2]), np.sort(_perm[nq // 2:])]

linhas, curvas = [], []
for nome, cfg in CONFIGS.items():
    t0 = time.time()
    prep = _preparar(cfg)
    sistemas = {}
    for j, f in enumerate(cfg["fontes"]):
        rot = f"só {f}" + ("" if f == cfg["base"] else f" (no top-{cfg['N']} de {cfg['base']})")
        sistemas[rot] = _avaliar(prep, lambda fs, j=j: fs[j])
    if len(cfg["fontes"]) == 2:
        por_alpha = {a: _avaliar(prep, lambda fs, a=a: a * _z(fs[0]) + (1 - a) * _z(fs[1])) for a in ALPHAS}
        curvas += [{"fusão": nome, "alpha": a, "MRR@10": r[:, 0].mean()} for a, r in por_alpha.items()]
        cv, escolhidos = np.empty((nq, 3)), []
        for k in range(2):
            treino, teste = FOLDS[k], FOLDS[1 - k]
            a_k = max(ALPHAS, key=lambda a: por_alpha[a][treino, 0].mean())
            escolhidos.append(float(a_k)); cv[teste] = por_alpha[a_k][teste]
        sistemas[f"interpolação, α por validação cruzada {escolhidos}"] = cv
        a_or = max(ALPHAS, key=lambda a: por_alpha[a][:, 0].mean())
        sistemas[f"[otimista] melhor α no próprio teste = {a_or}"] = por_alpha[a_or]
    else:
        sistemas["soma de z-scores (pesos iguais)"] = _avaliar(prep, lambda fs: sum(_z(x) for x in fs))
    sistemas[f"RRF (k={RRF_K})"] = _avaliar(prep, lambda fs: _rrf(*fs))

    unicos = [k for k in sistemas if k.startswith("só ")]
    melhor = max(unicos, key=lambda k: sistemas[k][:, 0].mean())
    ref = sistemas[melhor][:, 0]
    for sname, r in sistemas.items():
        x = r[:, 0]; lo, hi = _ic(x); d = x - ref
        lin = {"fusão": nome, "sistema": sname, "MRR@10": x.mean(), "IC95": f"[{lo:.4f}, {hi:.4f}]",
               "nDCG@10": r[:, 1].mean(), "Recall@10": r[:, 2].mean(),
               "melhor fonte sozinha": melhor, "Δ vs melhor fonte": np.nan if sname == melhor else d.mean(),
               "vitórias": int((d > 0).sum()), "derrotas": int((d < 0).sum()), "p": np.nan}
        if not sname.startswith(("só ", "[otimista]")):
            lin["p"] = wilcoxon(x, ref).pvalue if np.any(d != 0) else 1.0
        linhas.append(lin)
    print(f"  {nome}: {time.time()-t0:.0f}s")

fus = pd.DataFrame(linhas)
m = fus.p.notna(); fus.loc[m, "p (Holm)"] = _holm(fus.loc[m, "p"])
fus["significativo (α=0,05)"] = fus["p (Holm)"] < 0.05
fus.to_csv(f"{RES}/fusao_resultados.csv", index=False)
curvas = pd.DataFrame(curvas); curvas.to_csv(f"{RES}/fusao_curva_alpha.csv", index=False)
display(fus.drop(columns=["melhor fonte sozinha"]).round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
for nome, g in curvas.groupby("fusão", sort=False):
    ax.plot(g.alpha, g["MRR@10"], marker="o", label=nome)
ax.set_xlabel("α (peso da 1ª fonte: 1 = só ela, 0 = só a 2ª)"); ax.set_ylabel("MRR@10 (dev, todas as perguntas)")
ax.set_title("Fusão por interpolação de z-scores"); ax.grid(alpha=.3); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(f"{RES}/fusao_curva_alpha.png", dpi=200); plt.show()

print("Custo: a fusão em si é desprezível (somar vetores), mas paga a latência de TODAS as fontes. "
      "CE + monoPTT5 custa o mesmo que a cascata; CE pronto + CE nosso custa ~2x um CE; BM25 + CE custa o mesmo que o CE.")